# SN-01 — Sirenisation Phase 1 (SIREN exact)

Pour chaque EJ FINESS ayant un `nmsiren_stru`, lookup direct dans la base UL SIRENE complète. Cette phase produit notre **liste des SIREN validés** (VALIDE_FORT + VALIDE), qui sera utilisée pour exclure ces EJ du périmètre A/B/C.

Statuts : VALIDE_FORT / VALIDE / DOUTEUX / REJETE / SANS_SIREN / SIREN_INCONNU

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
from src.sirenisation import matching_direct_siren
from src.excel_export import export_phase1_excel, LABELS
from src.display      import afficher_tableau, afficher_synthese
from config.settings  import (
    FINESS_EJ_CLEAN, SIRENE_UL_CLEAN, SN_PHASE1, RESULTS_SN_DIR,
)

RESULTS_SN_DIR.mkdir(parents=True, exist_ok=True)

## 1. Chargement

In [2]:
df_ej = pd.read_parquet(FINESS_EJ_CLEAN)
df_ul = pd.read_parquet(SIRENE_UL_CLEAN)

df_ej['nmsiren_stru'] = df_ej['nmsiren_stru'].fillna('').astype(str)
df_ul['siren']        = df_ul['siren'].astype(str)

print(f'EJ FINESS : {len(df_ej):,}')
print(f'UL SIRENE : {len(df_ul):,}')

EJ FINESS : 54,097
UL SIRENE : 15,184,224


## 2. Matching SIREN exact

In [3]:
df_resultats = matching_direct_siren(df_ej, df_ul, desc='Matching SIREN exact')
print(df_resultats['statut'].value_counts())

Matching SIREN exact:   0%|          | 0/54097 [00:00<?, ?it/s]

statut
VALIDE_FORT      21042
VALIDE           17236
DOUTEUX           6805
SIREN_INCONNU     4031
REJETE            2885
SANS_SIREN        2098
Name: count, dtype: int64


## 3. Enrichissement avec colonnes UL SIRENE

In [4]:
COLS_UL_JOIN = ['siren', 'denominationUniteLegale', 'sigleUniteLegale',
                'categorieJuridiqueUniteLegale', 'activitePrincipaleUniteLegale',
                'dateCreationUniteLegale',
                'adresse_siege_complete_ul', 'codeCommuneEtablissement']
ul_join = df_ul[[c for c in COLS_UL_JOIN if c in df_ul.columns]].drop_duplicates('siren').copy()
ul_join['siren'] = ul_join['siren'].astype(str)

df_resultats['siren_ul'] = df_resultats['siren_ul'].astype(str)
df_resultats = df_resultats.merge(
    ul_join, left_on='siren_ul', right_on='siren', how='left',
).drop(columns=['siren'], errors='ignore')

## 4. Aperçu

In [5]:
afficher_tableau(
    df_resultats[df_resultats['statut'].isin(['VALIDE_FORT', 'VALIDE'])],
    'Aperçu validés Phase 1', max_lignes=9,
    colonnes=['idstructure_stru', 'nmsiren_stru', 'raisonsociale_stru',
              'denominationUniteLegale', 'nom_ul_retenu',
              'score_nom', 'score_adresse', 'score_global', 'statut'],
)

idstructure_stru,nmsiren_stru,raisonsociale_stru,denominationUniteLegale,nom_ul_retenu,score_nom,score_adresse,score_global,statut
1843334,424466183,COLLECTIF ASSOCIATIF DU BASSIN ALESIEN,COLLECTIF ASSOCIATIF DU BASSIN ALESIEN,COLLECTIF ASSOCIATIF BASSIN ALESIEN,100.000000,100.000000,100.000000,VALIDE_FORT
1843335,494422058,PHARMACIE CHAINIEUX,PHARMACIE CHAINIEUX,PHARMACIE CHAINIEUX,100.000000,100.000000,100.000000,VALIDE_FORT
1843336,953175106,PHARMACIE DE SAINT GERVASY,PHARMACIE DE SAINT GERVASY,PHARMACIE SAINT GERVASY,100.000000,100.000000,100.000000,VALIDE_FORT
1843351,263000564,CCAS LE GRAU DU ROI,CTRE COM ACTION SOCIALE DU GRAU DU ROI,CTRE COM ACTION SOCIALE GRAU ROI,54.320000,70.000000,63.730000,VALIDE
1843358,377919998,SARL LA DESIRADE,SARL LA DESIRADE,DESIRADE,100.000000,70.000000,82.000000,VALIDE
1843359,389159005,ESPACE SOCIAL,ESPACE SOCIAL,ESPACE SOCIAL,100.000000,100.000000,100.000000,VALIDE_FORT
1843360,388607012,ASSOC LA VIE EN DOUCE,ASS LA VIE EN DOUCE,ASS VIE DOUCE,85.430000,88.570000,87.310000,VALIDE_FORT
1843372,489388751,ASSOC SAMDO POMAREDE,SAMDO POMAREDE,SAMDO POMAREDE,77.650000,57.330000,65.460000,VALIDE
1843383,193000262,LYCEE DHUODA,LYCEE GENERAL TECHNO DIT DHUODA,LYCEE GENERAL TECHNO DIT DHUODA,52.550000,86.670000,73.020000,VALIDE


## 5. Export Excel

In [6]:
COLS_COMPLET = [
    'idstructure_stru', 'nmfinessej_stru', 'nmfinessetab_stru', 
    'categetab_stru', 'nmsiren_stru', 'raisonsociale_stru',
    'cdape_stru', 'dtouvertstruct_stru',
    'cdcommune_stru', 'adresse_complete_ej',
    'siren_ul', 'denominationUniteLegale', 'sigleUniteLegale',
    'nom_ul_retenu', 'adresse_siege_complete_ul', 'codeCommuneEtablissement',
    'categorieJuridiqueUniteLegale', 'activitePrincipaleUniteLegale',
    'dateCreationUniteLegale',
    'score_nom', 'score_adresse', 'score_global',
]
COLS_INFO = [
    'idstructure_stru', 'nmsiren_stru', 'raisonsociale_stru',
    'cdape_stru', 'dtouverture_stru',
    'cdcommune_stru', 'adresse_complete_ej',
]

compteurs = export_phase1_excel(
    df_resultats, SN_PHASE1, COLS_COMPLET, COLS_INFO,
    statuts_score=['VALIDE_FORT', 'VALIDE', 'DOUTEUX', 'REJETE'],
    statuts_info =['SANS_SIREN', 'SIREN_INCONNU'],
)

afficher_synthese({LABELS[s]: n for s, n in compteurs.items()},
                  'Synthèse Sirenisation Phase 1')
print(f'\nFichier : {SN_PHASE1}')

Statut,Nb,% du total
Valide_fort,"21,042",38.9%
Valide,"17,236",31.9%
Douteux,"6,805",12.6%
Rejeté,"2,885",5.3%
Sans_SIREN,"2,098",3.9%
SIREN_inconnu,"4,031",7.5%
TOTAL,"54,097",100.0%



Fichier : /home/jovyan/work/projet_finess_sirene/results/sirenisation/sirenisation_phase1.xlsx
